# 중심성 지표 가져오기 : central_df

In [1]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gustjs21@"))

def run_query(tx, query):
    return tx.run(query).data()

# 중심성 분석 알고리즘
algorithms = {
    "degree": "gds.degree.write",
    "betweenness": "gds.betweenness.write",
    "closeness": "gds.closeness.write",
    "eigenvector": "gds.eigenvector.write",
    "pagerank": "gds.pageRank.write",
    "community": "gds.louvain.write"
}

with driver.session(database="sicpama") as session:
    # 모든 DINED_WITH_* 관계 불러오기
    rels = session.execute_read(run_query, """
        CALL db.relationshipTypes() YIELD relationshipType
        WHERE relationshipType STARTS WITH 'DINED_WITH_'
        RETURN relationshipType
    """)
    
    for rel_obj in rels:
        rel_type = rel_obj['relationshipType']
        graph_name = f"graph_{rel_type}"

        print(f"\n🔄 {rel_type} 처리 시작...")

        # 기존 그래프가 있다면 삭제
        try:
            drop_query = f"""
            CALL gds.graph.exists('{graph_name}') YIELD exists
            WITH exists WHERE exists
            CALL gds.graph.drop('{graph_name}') YIELD graphName
            RETURN graphName
            """
            session.execute_write(run_query, drop_query)
            print(f"🧹 이전 그래프 삭제: {graph_name}")
        except:
            pass

        # projection 수행
        try:
            projection_query = f"""
            CALL gds.graph.project.cypher(
              '{graph_name}',
              'MATCH (c:Customer)-[:{rel_type}]-() RETURN id(c) AS id',
              'MATCH (c1:Customer)-[r:{rel_type}]->(c2:Customer)
               RETURN id(c1) AS source, id(c2) AS target, r.times AS times'
            )
            """
            session.execute_write(run_query, projection_query)
            print(f"✅ Projection 완료: {graph_name}")
        except Exception as e:
            print(f"❌ Projection 실패 for {rel_type}: {e}")
            continue  # 다음 store로 넘어감

        # 중심성 분석 수행
        for algo, func in algorithms.items():
            prop_name = f"{algo}_{rel_type}"
            print(f"→ {algo} 계산 중...")

            try:
                algo_query = f"""
                CALL {func}('{graph_name}', {{
                  writeProperty: '{prop_name}'
                }})
                """
                session.execute_write(run_query, algo_query)
                print(f"✅ {algo} 완료 → {prop_name}")
            except Exception as e:
                print(f"❌ {algo} 실패: {e}")


🔄 DINED_WITH_1 처리 시작...
🧹 이전 그래프 삭제: graph_DINED_WITH_1


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('gds.graph.project.cypher' has been replaced by 'gds.graph.project Cypher projection as an aggregation function')} {position: line: 2, column: 13, offset: 13} for query: "\n            CALL gds.graph.project.cypher(\n              'graph_DINED_WITH_1',\n              'MATCH (c:Customer)-[:DINED_WITH_1]-() RETURN id(c) AS id',\n              'MATCH (c1:Customer)-[r:DINED_WITH_1]->(c2:Customer)\n               RETURN id(c1) AS source, id(c2) AS target, r.times AS times'\n            )\n            "


✅ Projection 완료: graph_DINED_WITH_1
→ degree 계산 중...
✅ degree 완료 → degree_DINED_WITH_1
→ betweenness 계산 중...
✅ betweenness 완료 → betweenness_DINED_WITH_1
→ closeness 계산 중...
✅ closeness 완료 → closeness_DINED_WITH_1
→ eigenvector 계산 중...
✅ eigenvector 완료 → eigenvector_DINED_WITH_1
→ pagerank 계산 중...
✅ pagerank 완료 → pagerank_DINED_WITH_1
→ community 계산 중...
✅ community 완료 → community_DINED_WITH_1

🔄 DINED_WITH_3 처리 시작...
🧹 이전 그래프 삭제: graph_DINED_WITH_3


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('gds.graph.project.cypher' has been replaced by 'gds.graph.project Cypher projection as an aggregation function')} {position: line: 2, column: 13, offset: 13} for query: "\n            CALL gds.graph.project.cypher(\n              'graph_DINED_WITH_3',\n              'MATCH (c:Customer)-[:DINED_WITH_3]-() RETURN id(c) AS id',\n              'MATCH (c1:Customer)-[r:DINED_WITH_3]->(c2:Customer)\n               RETURN id(c1) AS source, id(c2) AS target, r.times AS times'\n            )\n            "


✅ Projection 완료: graph_DINED_WITH_3
→ degree 계산 중...
✅ degree 완료 → degree_DINED_WITH_3
→ betweenness 계산 중...
✅ betweenness 완료 → betweenness_DINED_WITH_3
→ closeness 계산 중...
✅ closeness 완료 → closeness_DINED_WITH_3
→ eigenvector 계산 중...
✅ eigenvector 완료 → eigenvector_DINED_WITH_3
→ pagerank 계산 중...
✅ pagerank 완료 → pagerank_DINED_WITH_3
→ community 계산 중...
✅ community 완료 → community_DINED_WITH_3

🔄 DINED_WITH_4 처리 시작...
🧹 이전 그래프 삭제: graph_DINED_WITH_4


...

✅ eigenvector 완료 → eigenvector_DINED_WITH_17
→ pagerank 계산 중...
✅ pagerank 완료 → pagerank_DINED_WITH_17
→ community 계산 중...
✅ community 완료 → community_DINED_WITH_17

🔄 DINED_WITH_10035 처리 시작...
🧹 이전 그래프 삭제: graph_DINED_WITH_10035
✅ Projection 완료: graph_DINED_WITH_10035
→ degree 계산 중...
✅ degree 완료 → degree_DINED_WITH_10035
→ betweenness 계산 중...
✅ betweenness 완료 → betweenness_DINED_WITH_10035
→ closeness 계산 중...
✅ closeness 완료 → closeness_DINED_WITH_10035
→ eigenvector 계산 중...
✅ eigenvector 완료 → eigenvector_DINED_WITH_10035
→ pagerank 계산 중...
✅ pagerank 완료 → pagerank_DINED_WITH_10035
→ community 계산 중...
✅ community 완료 → community_DINED_WITH_10035

🔄 DINED_WITH_10039 처리 시작...
🧹 이전 그래프 삭제: graph_DINED_WITH_10039


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('gds.graph.project.cypher' has been replaced by 'gds.graph.project Cypher projection as an aggregation function')} {position: line: 2, column: 13, offset: 13} for query: "\n            CALL gds.graph.project.cypher(\n              'graph_DINED_WITH_10039',\n              'MATCH (c:Customer)-[:DINED_WITH_10039]-() RETURN id(c) AS id',\n              'MATCH (c1:Customer)-[r:DINED_WITH_10039]->(c2:Customer)\n               RETURN id(c1) AS source, id(c2) AS target, r.times AS times'\n            )\n            "


✅ Projection 완료: graph_DINED_WITH_10039
→ degree 계산 중...
✅ degree 완료 → degree_DINED_WITH_10039
→ betweenness 계산 중...
✅ betweenness 완료 → betweenness_DINED_WITH_10039
→ closeness 계산 중...
✅ closeness 완료 → closeness_DINED_WITH_10039
→ eigenvector 계산 중...
✅ eigenvector 완료 → eigenvector_DINED_WITH_10039
→ pagerank 계산 중...
✅ pagerank 완료 → pagerank_DINED_WITH_10039
→ community 계산 중...
✅ community 완료 → community_DINED_WITH_10039

🔄 DINED_WITH_10042 처리 시작...
🧹 이전 그래프 삭제: graph_DINED_WITH_10042


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('gds.graph.project.cypher' has been replaced by 'gds.graph.project Cypher projection as an aggregation function')} {position: line: 2, column: 13, offset: 13} for query: "\n            CALL gds.graph.project.cypher(\n              'graph_DINED_WITH_10042',\n              'MATCH (c:Customer)-[:DINED_WITH_10042]-() RETURN id(c) AS id',\n              'MATCH (c1:Customer)-[r:DINED_WITH_10042]->(c2:Customer)\n               RETURN id(c1) AS source, id(c2) AS target, r.times AS times'\n            )\n            "


✅ Projection 완료: graph_DINED_WITH_10042
→ degree 계산 중...
✅ degree 완료 → degree_DINED_WITH_10042
→ betweenness 계산 중...
✅ betweenness 완료 → betweenness_DINED_WITH_10042
→ closeness 계산 중...
✅ closeness 완료 → closeness_DINED_WITH_10042
→ eigenvector 계산 중...
✅ eigenvector 완료 → eigenvector_DINED_WITH_10042
→ pagerank 계산 중...
✅ pagerank 완료 → pagerank_DINED_WITH_10042
→ community 계산 중...
✅ community 완료 → community_DINED_WITH_10042


# orders 정보 가져오기 : visit_df

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# ── 0. MySQL 접속 ────────────────────
# 실제 접속 정보로 아래 값을 수정하세요.
DB_USER = "dev"
DB_PASS = "pwd"
DB_HOST = "localhost"
DB_PORT = "3306"
DB_NAME = "orders"

engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# ── 제외할 customerId 목록 ──────────────────────────────
excluded_ids = [
    '01H8K9XWK972V31SN3HDRGY022','01H7S6309C3ZWPVVMAWDYQDQE9',
    '01H7JEGZX7C0NCPN8ZZ3AJXASE','01H7YB33EPCWHTX1WWA9ANJCPG',
    '01H7VSQW5TB8Q4HR1Z4S5CTJKQ','01H4001EP0F1859987V7D4YJFG',
    '01H8DZZ71SYRKB4868B0WHV68E','01H98VZ8Y98MJ489TEJW7QCDS4',
    '01H7S0P3HK62VATNYDHADQPHKJ','01H9A1GTJVCTA13SA9YPEB9ETA',
    '01H7QY348FZQRVP21JAN8X4Z7R','01H7S9MDFE2FW13DPP93DV0WWP',
    '01H7SVW6ECGZ5JXPSV0YS6P7KG','01H7J0A52BKC867SHMNBZ85XTC',
    '01H847J2DGX5KEKMX386TKYGY1','01H7HNC3GKV65TAG2H3SGW7A0H',
    '01H98W8WA1X2E09TZ50DHHRJSA','01H59HRM9W7WZC0Y0Y5YK4ZJS8',
    '01H7HTY1JNFB3H0MP72X5R4QJW','01H7QXJZ7MQ63FTGJTJ5WB5TFR',
    '01H98VE7QSYH0W7AFH9Z9RY6KZ','01H3ZY9J55GM6S18YQGR3DCGQT',
    '01H8Y02HDQZTAH5AVKP51RN05G','01H847HNN73J8V4DN1V1Z7QJTT',
    '01GZZ0C5HVG9049C8AT5Y96HFS','01H9MSJKX4CCDBG2JK327BSP84',
    '01H9MAY8ZDX7KMFBS7EJYJR5AF','01HFDTAW1QRVR8P1HDKH4CNM47',
    '01HF8FYGWYH8Y27TMDPDAK2MH1','01HDJHZ6WM6QQAG627YQ549S8Z',
    '01HF3DR2VBV3D2M0SB6M32WXHQ', '01GYNFXQNDD0ATWHDFXCBHD5XG',
    '01H9CY93FVA1J4SA755K6JNAQ8', '01H9W8SZ19CAVG54Z4XGR2DWTR'
]
excluded_ids_str = ', '.join(f"'{cid}'" for cid in excluded_ids)

# ── 1. 방문 단위 데이터 (세션별 최초 주문) ────────
query_visits = f"""
    SELECT
        customerId,
        sessionId,
        MIN(storeId)   AS storeId,
        MIN(createdAt) AS createdAt
    FROM orders
    WHERE customerId NOT IN ({excluded_ids_str})
    GROUP BY customerId, sessionId
"""
visits = pd.read_sql(query_visits, engine, parse_dates=["createdAt"])

# ── 2. 고객 × 매장 단위 방문 분석 ─────────────
def avg_interval(series):
    if len(series) < 2:
        return None
    s = series.sort_values()
    return (s.diff().dt.total_seconds() / 86_400).mean()

agg = (
    visits.groupby(["customerId", "storeId"])
          .agg(visit_count=('createdAt', 'size'),
               first_visit=('createdAt', 'min'),
               last_visit=('createdAt', 'max'),
               avg_revisit_days=('createdAt', avg_interval))
          .reset_index()
)

# ── 3. 주요 매장 선정 (방문 수 최다) ─────────────
top_idx = agg.groupby("customerId")["visit_count"].idxmax()
main_store = (
    agg.loc[top_idx]
       .rename(columns={"storeId": "main_storeId",
                        "visit_count": "main_store_visits",
                        "avg_revisit_days": "main_store_avg_revisit_days"})
)

# ── 4. 고객 요약 테이블 ────────────────────
visit_df = (
    visits.groupby("customerId")
          .size()
          .reset_index(name="total_visits")
          .merge(main_store, on="customerId")
          .sort_values("total_visits", ascending=False)
          .reset_index(drop=True)
)

# ── 5. 결과 확인 ────────────────────────
display(visit_df.head(50))

# join으로 통합

In [ ]:
# visit_df와 customerId 기준으로 join
customers_df = customers_df.merge(visit_df, on="customerId", how="right")

In [ ]:
customers_df

In [ ]:
customers_df = customers_df = customers_df.merge(central_df, on="customerId", how="right")

In [ ]:
customers_df.info()

In [ ]:
cus_marketing_df = customers_df[
    ~customers_df['isMarketingAgreed'].isna()
]
cus_marketing_df

In [ ]:
cus_marketing_df.info()

In [ ]:
cus_marketing_score_df = cus_marketing_df.drop(['phoneNumber','dateOfBirth','isMarketingAgreed','first_visit','last_visit','community'], axis=1)
cus_marketing_score_df.head(20)

In [ ]:
cus_marketing_score_df.to_csv('../data/복합점수준비.csv', index=False)